# CS421 Phase II — Image Classification Robustness (Group 2)

Runs the full experiment on Google Colab.

**Before starting:** Runtime → Change runtime type → **T4 GPU**.

Checkpoints and results are saved to Google Drive, so if Colab disconnects, just run all cells again: finished runs are skipped and unfinished runs resume from their last epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
# (go to a folder that always exists before deleting and re-cloning the repo;
#  otherwise re-running this cell fails with 'Unable to read current working directory')

REPO = 'https://github.com/AfnanNas379/image-classification-robustness.git'
!rm -rf /content/repo && git clone -q $REPO /content/repo
%cd /content/repo
!pip -q install -r requirements.txt

# Keep data, checkpoints and results on Drive so they survive disconnects
DRIVE = '/content/drive/MyDrive/CS421_phase2'
!mkdir -p $DRIVE/data $DRIVE/runs $DRIVE/results
!ln -sfn $DRIVE/data data && ln -sfn $DRIVE/runs runs && ln -sfn $DRIVE/results results

## 2. Generate the 13 fixed test sets (once, ~3 min)
Prints mean PSNR and SSIM per condition and writes `data/degraded/manifest.json` with SHA-256 checksums.

In [ ]:
import os
if not os.path.exists('data/degraded/manifest.json'):
    !python -m scripts.make_degraded
else:
    print('already generated')
!cat data/degraded/manifest.json | head -40

## 3. Train: 3 models × 3 seeds
About 15 min (Plain CNN), 35–45 min (ResNet-18) and 40–50 min (ViT) per run on a T4. You can split this over several sessions or team members' Colab accounts, as long as everyone uses the **same Drive folder** (share `CS421_phase2`).

In [ ]:
MODELS = ['plain_cnn', 'resnet18', 'vit']
SEEDS = [0, 1, 2]
for seed in SEEDS:
    for m in MODELS:
        !python -m src.train --model {m} --seed {seed} --out runs

## 4. Evaluate on all 13 conditions

In [ ]:
!python -m src.evaluate --runs runs --out results

## 5. Figures (300 dpi PNG + PDF, greyscale-readable)

In [ ]:
!python -m src.plots --results results --runs runs --degraded data/degraded --out results/figures
from IPython.display import Image, display
for f in ['accuracy_vs_severity', 'rd_per_type', 'ece_vs_severity', 'training_curves', 'degradation_examples']:
    display(Image(f'results/figures/{f}.png'))

## 6. Download results to commit to GitHub
Commit `results/*.csv`, `results/tables.tex`, `results/figures/`, `data/degraded/manifest.json` and each run's `log.csv`, `meta.json`, `done.json`. Do **not** commit the `.pt` checkpoints or the `.npz` image sets (too large).

In [ ]:
!mkdir -p /content/export/runs
!cp -r results /content/export/ && cp data/degraded/manifest.json /content/export/
!for d in runs/*_seed*; do mkdir -p /content/export/$d; cp $d/*.csv $d/*.json /content/export/$d/; done
!cd /content && zip -qr phase2_results.zip export
from google.colab import files
files.download('/content/phase2_results.zip')